## SMILE-CR Dataset Preprocessing

1. Download the SMILE-CR dataset from [this link](https://www.kaggle.com/api/v1/datasets/download/yuxiawhu/smile-cr).
2. Unzip the smile-cr.zip file. The unzipped folders should look like this:
    ```
    |- TestData/
          |- TestData
                  |- CloudLandsat_2020
                  |- Landsat-8_2020
                  |- MODIS_2020
                  |- Mask
                  |- Sentinel-1_2020-De
    |- TrainData/
    |- ValData
    ```
3. Run the code blocks below to preprocess the dataset and obtain the train/val/test splits in .npy format.
4. Go to `./CycleGAN` and run `CycleGAN.ipynb` notebook to obtain translated images.

The data folder after finishing preprocessing should look like this:

        SMILE-CR-npy/
            |- TestData/
                |- TestData
                        |- CloudLandsat_2020
                        |- Landsat-8_2020
                        |- MODIS_2020
                        |- Mask
                        |- Sentinel-1_2020-De
                        |- SMILECR_translated_npy
            |- TrainData/
            |- ValData


In [ ]:
!curl -L -o smile-cr.zip https://www.kaggle.com/api/v1/datasets/download/yuxiawhu/smile-cr

In [ ]:
!unzip smile-cr.zip

In [1]:
import os
import shutil
import rasterio
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

def convert_smilecr_to_npy(source_root, target_root, splits=['train', 'val', 'test']):
    """
    Convert SMILE-CR TIFF dataset to NPY format while maintaining exact structure and filenames.
    
    Args:
        source_root: Path to original SMILE-CR dataset
        target_root: Path where NPY version will be created
        splits: List of splits to convert ['train', 'val', 'test']
    """
    
    # Suppress rasterio warnings
    warnings.filterwarnings('ignore', category=UserWarning, module='rasterio')
    
    source_root = Path(source_root)
    target_root = Path(target_root)
    
    # Create target directory
    target_root.mkdir(exist_ok=True)
    
    # Define split mappings
    split_mapping = {
        'train': ('TrainData', 'TrainData'),
        'val': ('ValData', 'ValData'), 
        'test': ('TestData', 'TestData')
    }
    
    # Subdirectories to process
    subdirs = ['CloudLandsat_2020', 'Landsat-8_2020', 'MODIS_2020', 'Mask', 'Sentinel-1_2020-De']
    
    total_files = 0
    converted_files = 0
    
    for split in splits:
        if split not in split_mapping:
            print(f"Unknown split: {split}")
            continue
            
        source_split_dir = source_root / split_mapping[split][0] / split_mapping[split][1]
        target_split_dir = target_root / f"{split.capitalize()}Data" / f"{split.capitalize()}Data"
        
        if not source_split_dir.exists():
            print(f"Source directory not found: {source_split_dir}")
            continue
            
        print(f"\nProcessing {split} split...")
        print(f"Source: {source_split_dir}")
        print(f"Target: {target_split_dir}")
        
        # Create target split directory
        target_split_dir.mkdir(parents=True, exist_ok=True)
        
        for subdir in subdirs:
            source_subdir = source_split_dir / subdir
            target_subdir = target_split_dir / subdir
            
            if not source_subdir.exists():
                print(f"  Skipping {subdir} (not found)")
                continue
                
            # Create target subdirectory
            target_subdir.mkdir(exist_ok=True)
            
            # Get all TIFF files in this subdirectory
            tiff_files = list(source_subdir.glob("*.tif"))
            
            if not tiff_files:
                print(f"  {subdir}: No TIFF files found")
                continue
                
            print(f"  {subdir}: Converting {len(tiff_files)} files...")
            total_files += len(tiff_files)
            
            # Convert each TIFF file to NPY
            for tiff_file in tqdm(tiff_files, desc=f"    {subdir}"):
                try:
                    # Read TIFF file
                    with rasterio.open(tiff_file) as src:
                        data = src.read()  # Shape: (bands, height, width)
                        data = data.astype(np.float32)
                    
                    # Create corresponding NPY filename
                    npy_filename = tiff_file.stem + ".npy"
                    npy_path = target_subdir / npy_filename
                    
                    # Save as NPY
                    np.save(npy_path, data)
                    converted_files += 1
                    
                except Exception as e:
                    print(f"    Error converting {tiff_file.name}: {e}")
            
            print(f"    Completed {subdir}")
    
    print(f"\nConversion Summary:")
    print(f"Total files found: {total_files}")
    print(f"Successfully converted: {converted_files}")
    print(f"Failed conversions: {total_files - converted_files}")
    print(f"NPY dataset saved to: {target_root}")


In [ ]:
# Execute the conversion
source_root = "./SMILE-CR"
target_root = "./SMILE-CR-npy"

print("Starting SMILE-CR to NPY conversion...")
convert_smilecr_to_npy(source_root, target_root, splits=['train', 'val', 'test'])